In [0]:
from pyspark.sql import functions as f
import sys
sys.path.append('..')
sys.path.append('../..')

import lib_dna_member.job_manager as job_manager
import lib_dna_member.transaction_features as features
from lib_dna_member.s3 import member_dna_input_data_validator
from databricks.feature_engineering import FeatureEngineeringClient

In [0]:
%run ../../config/utils

In [0]:
def generate_transaction(job):
    """
    Generate transaction based variables for the given population
    Parameters:
        job (object): Job Manager object based on the current config file

    Returns:
        dna (pyspark.sql.DataFrame): Transaction data of the given population
    """
    dna = job.tables["cubes_transaction_1"]
    orig_cols = dna.columns

    num_weeks = ["FOUR", "EIGHT", "TWELVE", "TWENTY-SIX", "FIFTY-TWO"]

    dna = features.feature_agg_spend(job, dna, num_weeks)

    dna = features.feature_agg_spend_in_store(job, dna, num_weeks)

    dna = features.feature_agg_units(job, dna, num_weeks)

    dna = features.feature_agg_units_over_fifty(job, dna, num_weeks)

    dna = features.feature_agg_gas_trips(job, dna, num_weeks)

    dna = features.feature_agg_gas_distinct_days(job, dna, num_weeks)

    dna = features.feature_agg_gas_spend(job, dna, num_weeks)

    dna = features.feature_agg_gas_and_store_distinct_days(job, dna, num_weeks)

    dna = features.feature_agg_ecommerce_metric(job, dna, num_weeks, "spend")

    dna = features.feature_agg_ecommerce_metric(job, dna, num_weeks, "trips")

    dna = features.feature_agg_transactions(job, dna, num_weeks)

    dna = dna.drop(
        *[
            col
            for col in orig_cols
            if col not in ["MBRSHP_SID", "FISCAL_WEEK_END"]
        ]
    )

    return dna

In [0]:
job = job_manager.JobManager(spark, intermediate_all_tables_dict, member_dna_config_path)

In [0]:
recency_lookback_duration = job.config["params"].get( 
    "recency_lookback_duration", {}
)
member_dna_input_data_validator(
    fs_cubes_transaction_1,
    recency_lookback_duration=recency_lookback_duration,
    spark=spark
)



In [0]:
job.read_table("cubes_transaction_1")

In [0]:
features = generate_transaction(job)

### Save results

In [0]:
spark.sql(f"DELETE FROM {fs_cubes_transaction_2}")

fe = FeatureEngineeringClient()

fe.write_table(
    name=fs_cubes_transaction_2,
    df=features,
    mode="merge"
)